# Experiment 1

As the dataset is a tabular data, I'm going to try bunch of experimetns using MLP

In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, roc_auc_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

## Import data

Make train and test features and labels dataframes

In [ ]:
DATA_DIR = "../data"

# --------------  Train Features and Labels ------------------
main_dataset = pd.read_csv(f"{DATA_DIR}/train_features.csv")
main_dataset.set_index("sig_id", inplace=True)  # set sig_id as index

# filter out control samples
main_dataset = main_dataset[main_dataset["cp_type"] == "trt_cp"]
main_dataset["cp_dose_bin"] = main_dataset["cp_dose"].map({"D1": 0, "D2": 1})  # make cp_dose a binary variable

# make metadata dataframe
metadata_cols = ["cp_type", "cp_time", "cp_dose"]
main_metadata = main_dataset[metadata_cols].copy()
# train_metadata_df.set_index("sig_id", inplace=True)  # set sig_id as index

# make train_features dataframe
feature_cols = [col for col in main_dataset.columns if col not in metadata_cols and col != "sig_id"]
main_features_df = main_dataset[feature_cols].copy()


# Make train_targets dataframe
main_targets_df = pd.read_csv(f"{DATA_DIR}/train_targets_scored.csv")
main_targets_df.set_index("sig_id", inplace=True)  # set sig_id as index
main_targets_df

/tmp/ipykernel_3645798/3466063986.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_df["cp_dose_bin"] = train_df["cp_dose"].map({"D1": 0, "D2": 1})  # make cp_dose a binary variable


,5-alpha_reductase_inhibitor,11-beta-hsd1_inhibitor,acat_inhibitor,acetylcholine_receptor_agonist,acetylcholine_receptor_antagonist,acetylcholinesterase_inhibitor,adenosine_receptor_agonist,adenosine_receptor_antagonist,adenylyl_cyclase_activator,adrenergic_receptor_agonist,...,tropomyosin_receptor_kinase_inhibitor,trpv_agonist,trpv_antagonist,tubulin_inhibitor,tyrosine_kinase_inhibitor,ubiquitin_specific_protease_inhibitor,vegfr_inhibitor,vitamin_b,vitamin_d_receptor_agonist,wnt_inhibitor
sig_id,,,,,,,,,,,,,,,,,,,,,
id_000644bb2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
id_000779bfc,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
id_000a6266a,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
id_0015fd391,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
id_001626bd3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
id_fffb1ceed,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
id_fffb70c0c,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
id_fffc1c3f4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
DATA_DIR = "../data"

# --------------  Train Features and Labels ------------------
main_dataset = pd.read_csv(f"{DATA_DIR}/train_features.csv")
main_dataset.set_index("sig_id", inplace=True)  # set sig_id as index

# filter out control samples
main_dataset = main_dataset[main_dataset["cp_type"] == "trt_cp"]
main_dataset["cp_dose_bin"] = main_dataset["cp_dose"].map({"D1": 0, "D2": 1})  # make cp_dose a binary variable

# make metadata dataframe
metadata_cols = ["cp_type", "cp_time", "cp_dose"]
main_metadata = main_dataset[metadata_cols].copy()
# train_metadata_df.set_index("sig_id", inplace=True)  # set sig_id as index

# make train_features dataframe
feature_cols = [col for col in main_dataset.columns if col not in metadata_cols and col != "sig_id"]
main_features_df = main_dataset[feature_cols].copy()


# Make train_targets dataframe
main_targets_df = pd.read_csv(f"{DATA_DIR}/train_targets_scored.csv")
main_targets_df.set_index("sig_id", inplace=True)  # set sig_id as index

# -------------- Test Features and Labels -----------------
# Make test_features dataframe
test_features_df = pd.read_csv(f"{DATA_DIR}/test_features.csv")
test_features_df.set_index("sig_id", inplace=True)  # set sig_id as index

# filter out control samples
test_features_df = test_features_df[test_features_df["cp_type"] == "trt_cp"]
test_features_df["cp_dose_bin"] = test_features_df["cp_dose"].map({"D1": 0, "D2": 1})  # make cp_dose a binary variable

# Make test_metadata dataframe
test_metadata_df = test_features_df[metadata_cols].copy()

# make test_features dataframe
test_features_df = test_features_df[feature_cols].copy()

/tmp/ipykernel_3645798/847061885.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_df["cp_dose_bin"] = train_df["cp_dose"].map({"D1": 0, "D2": 1})  # make cp_dose a binary variable
/tmp/ipykernel_3645798/847061885.py:32: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test_features_df["cp_dose_bin"] = test_features_df["cp_dose"].map({"D1": 0, "D2": 1})  # make cp_dose a binary variable


FileNotFoundError: [Errno 2] No such file or directory: '../data/test_targets_scored.csv'

## Train / Val / Dev Split

Split labeled training data into train (~60%), val (~20%), dev (~20%) using `MultilabelStratifiedKFold` (5 folds). Since each fold's held-out indices are non-overlapping and cover the full dataset, we assign fold 0 → val, fold 1 → dev, folds 2–4 → train.

In [ ]:
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold

# Align features and targets on shared sig_ids (main_features_df is already filtered to trt_cp)
common_idx = main_features_df.index.intersection(main_targets_df.index)
X = main_features_df.loc[common_idx]
y = main_targets_df.loc[common_idx]

mskf = MultilabelStratifiedKFold(n_splits=5, shuffle=True, random_state=42)
splits = list(mskf.split(X, y))

# Each fold's test indices are non-overlapping and together cover the full dataset.
# fold 0 → val (~20%), fold 1 → dev (~20%), folds 2-4 → train (~60%)
val_idx   = splits[0][1]
dev_idx   = splits[1][1]
train_idx = np.concatenate([splits[i][1] for i in range(2, 5)])

X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
X_val,   y_val   = X.iloc[val_idx],   y.iloc[val_idx]
X_dev,   y_dev   = X.iloc[dev_idx],   y.iloc[dev_idx]

print(f"Train: {len(X_train)} | Val: {len(X_val)} | Dev: {len(X_dev)}")

## PyTorch Dataset and DataLoaders

In [ ]:
class MoADataset(Dataset):
    def __init__(self, features, targets=None):
        self.X = torch.tensor(features.values, dtype=torch.float32)
        self.y = torch.tensor(targets.values, dtype=torch.float32) if targets is not None else None

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        if self.y is not None:
            return self.X[idx], self.y[idx]
        return self.X[idx]

In [ ]:
BATCH_SIZE = 128

train_dataset = MoADataset(X_train, y_train)
val_dataset   = MoADataset(X_val,   y_val)
dev_dataset   = MoADataset(X_dev,   y_dev)
test_dataset  = MoADataset(test_features_df)  # no labels — Kaggle test set

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
dev_loader   = DataLoader(dev_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Batches — train: {len(train_loader)} | val: {len(val_loader)} | dev: {len(dev_loader)} | test: {len(test_loader)}")

In [3]:
test_df = pd.read_csv(f"{DATA_DIR}/test_features.csv")
test_df

,sig_id,cp_type,cp_time,cp_dose,g-0,g-1,g-2,g-3,g-4,g-5,...,c-90,c-91,c-92,c-93,c-94,c-95,c-96,c-97,c-98,c-99
0,id_0004d9e33,trt_cp,24,D1,-0.5458,0.1306,-0.5135,0.4408,1.5500,-0.1644,...,0.0981,0.7978,-0.1430,-0.2067,-0.2303,-0.1193,0.0210,-0.0502,0.1510,-0.7750
1,id_001897cda,trt_cp,72,D1,-0.1829,0.2320,1.2080,-0.4522,-0.3652,-0.3319,...,-0.1190,-0.1852,-1.0310,-1.3670,-0.3690,-0.5382,0.0359,-0.4764,-1.3810,-0.7300
2,id_002429b5b,ctl_vehicle,24,D1,0.1852,-0.1404,-0.3911,0.1310,-1.4380,0.2455,...,-0.2261,0.3370,-1.3840,0.8604,-1.9530,-1.0140,0.8662,1.0160,0.4924,-0.1942
3,id_00276f245,trt_cp,24,D2,0.4828,0.1955,0.3825,0.4244,-0.5855,-1.2020,...,0.1260,0.1570,-0.1784,-1.1200,-0.4325,-0.9005,0.8131,-0.1305,0.5645,-0.5809
4,id_0027f1083,trt_cp,48,D1,-0.3979,-1.2680,1.9130,0.2057,-0.5864,-0.0166,...,0.4965,0.7578,-0.1580,1.0510,0.5742,1.0900,-0.2962,-0.5313,0.9931,1.8380
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3977,id_ff7004b87,trt_cp,24,D1,0.4571,-0.5743,3.3930,-0.6202,0.8557,1.6240,...,-1.1790,-0.6422,-0.4367,0.0159,-0.6539,-0.4791,-1.2680,-1.1280,-0.4167,-0.6600
3978,id_ff925dd0d,trt_cp,24,D1,-0.5885,-0.2548,2.5850,0.3456,0.4401,0.3107,...,0.0210,0.5780,-0.5888,0.8057,0.9312,1.2730,0.2614,-0.2790,-0.0131,-0.0934
3979,id_ffb710450,trt_cp,72,D1,-0.3985,-0.1554,0.2677,-0.6813,0.0152,0.4791,...,0.4418,0.9153,-0.1862,0.4049,0.9568,0.4666,0.0461,0.5888,-0.4205,-0.1504
3980,id_ffbb869f2,trt_cp,48,D2,-1.0960,-1.7750,-0.3977,1.0160,-1.3350,-0.2207,...,0.3079,-0.4473,-0.8192,0.7785,0.3133,0.1286,-0.2618,0.5074,0.7430,-0.0484
